# PTX-498 + NIH No Finding — Dataset Hazırlık
**TÜBİTAK 2209-A | Ahmet Demir**

Dataset:
- **Pozitif**: PTX-498 (498 vaka, radyolog onaylı, PNG + maske)
- **Negatif**: NIH Chest X-ray14 "No Finding" (~1500 vaka, temiz)


In [ ]:
# ── 1. Drive Bağla ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE  = Path('/content/drive/MyDrive/tubitak_pneumothorax')
PTX498_DIR  = DRIVE_BASE / 'data/ptx498'          # Mac'ten yüklediğin klasör
NIH_DIR     = DRIVE_BASE / 'data/nih_no_finding'  # NIH negatifler buraya
NIH_DIR.mkdir(parents=True, exist_ok=True)

print(f'PTX498 klasörü mevcut: {PTX498_DIR.exists()}')
if PTX498_DIR.exists():
    imgs = list(PTX498_DIR.glob('*.1.img.png'))
    print(f'PTX498 görüntü sayısı: {len(imgs)}')

In [ ]:
# ── 2. Kaggle API Kur ──────────────────────────────────────────────────────
# kaggle.com → Account → API → Create New Token → kaggle.json indir
from google.colab import files
print('kaggle.json dosyasını seç ve yükle:')
uploaded = files.upload()

import shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle API hazır')

In [ ]:
# ── 3. NIH CSV İndir → No Finding Filtrele ─────────────────────────────────
import subprocess, pandas as pd

os.makedirs('/content/nih_tmp', exist_ok=True)

print('📥 NIH CSV indiriliyor...')
subprocess.run([
    'kaggle', 'datasets', 'download',
    '-d', 'nih-chest-xrays/data',
    '--file', 'Data_Entry_2017.csv',
    '-p', '/content/nih_tmp/'
], check=True)
subprocess.run(['unzip', '-q', '-o',
    '/content/nih_tmp/Data_Entry_2017.csv.zip',
    '-d', '/content/nih_tmp/'], check=True)

df = pd.read_csv('/content/nih_tmp/Data_Entry_2017.csv')
no_finding = df[df['Finding Labels'] == 'No Finding'].copy()
no_finding = no_finding.sample(n=1500, random_state=42).reset_index(drop=True)
no_finding.to_csv('/content/nih_tmp/no_finding_1500.csv', index=False)

print(f'✅ No Finding toplam: {len(df[df["Finding Labels"] == "No Finding"])}')
print(f'✅ Seçilen: {len(no_finding)} vaka')
print(no_finding[['Image Index', 'Finding Labels']].head())

In [ ]:
# ── 4. NIH Görüntüleri ZIP'ten Toplu İndir ─────────────────────────────────
# NIH dataseti 12 ZIP dosyasına bölünmüş (images_001 - images_012)
# Her ZIP'i indir → hedef görüntüleri çıkar → ZIP sil → sonraki

import zipfile
from tqdm.notebook import tqdm

target_ids = set(no_finding['Image Index'].tolist())
already_done = {p.name for p in NIH_DIR.glob('*.png')}
remaining = target_ids - already_done
print(f'Hedef: {len(target_ids)} | Mevcut: {len(already_done)} | Kalan: {len(remaining)}')

for zip_idx in range(1, 13):
    zip_name = f'images_{zip_idx:03d}.zip'
    zip_path = f'/content/nih_tmp/{zip_name}'

    if not remaining:
        print('✅ Tüm görüntüler indirildi!')
        break

    print(f'\n📥 {zip_name} indiriliyor... (kalan hedef: {len(remaining)})')
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'nih-chest-xrays/data',
        '--file', zip_name,
        '-p', '/content/nih_tmp/'
    ], check=True)

    print(f'📦 {zip_name} açılıyor...')
    found_in_zip = 0
    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.namelist()
        for member in members:
            fname = os.path.basename(member)
            if fname in remaining:
                data = zf.read(member)
                out_path = NIH_DIR / fname
                with open(out_path, 'wb') as f:
                    f.write(data)
                remaining.discard(fname)
                found_in_zip += 1

    os.remove(zip_path)
    print(f'  ✓ {found_in_zip} görüntü kaydedildi → {zip_name} silindi')

final_count = len(list(NIH_DIR.glob('*.png')))
print(f'\n✅ NIH No Finding toplam: {final_count} görüntü → {NIH_DIR}')

In [ ]:
# ── 5. PTX-498 Kontrol ─────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

imgs   = sorted(PTX498_DIR.glob('*.1.img.png'))
masks  = sorted(PTX498_DIR.glob('*.2.mask.png'))
merges = sorted(PTX498_DIR.glob('*.3.merge.png'))

print(f'PTX-498:')
print(f'  Görüntü : {len(imgs)}')
print(f'  Maske   : {len(masks)}')
print(f'  Overlay : {len(merges)}')

# 6 örnek göster
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('PTX-498 — Örnek Vakalar (merge overlay)', fontsize=14)
for ax, p in zip(axes.flat, merges[:6]):
    img = Image.open(p)
    ax.imshow(img)
    ax.set_title(p.name[:20], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Dataset Özeti & Manifest Oluştur ───────────────────────────────────
import pandas as pd

records = []

# Pozitif: PTX-498
for img_path in sorted(PTX498_DIR.glob('*.1.img.png')):
    case_id = img_path.name.replace('.1.img.png', '')
    mask_path = PTX498_DIR / f'{case_id}.2.mask.png'
    records.append({
        'img_path'  : str(img_path),
        'mask_path' : str(mask_path) if mask_path.exists() else 'N/A',
        'is_pneumo' : 1,
        'source'    : 'PTX498',
        'case_id'   : case_id,
    })

# Negatif: NIH No Finding
for img_path in sorted(NIH_DIR.glob('*.png')):
    records.append({
        'img_path'  : str(img_path),
        'mask_path' : 'N/A',
        'is_pneumo' : 0,
        'source'    : 'NIH',
        'case_id'   : img_path.stem,
    })

manifest = pd.DataFrame(records)
manifest_path = DRIVE_BASE / 'data/manifest_ptx498_nih.csv'
manifest.to_csv(manifest_path, index=False)

print('='*50)
print('DATASET ÖZETİ')
print('='*50)
print(f'Pozitif (PTX-498) : {(manifest.is_pneumo==1).sum()}')
print(f'Negatif (NIH)     : {(manifest.is_pneumo==0).sum()}')
print(f'Toplam            : {len(manifest)}')
print(f'Manifest          : {manifest_path}')
print(f'\nEğitim komutu:')
print(f'python scripts/train_global.py \\')
print(f'  --manifest {manifest_path} \\')
print(f'  --sources PTX498,NIH \\')
print(f'  --encoder efficientnet-b4 \\')
print(f'  --epochs 50 --batch_size 16')

In [ ]:
# ── 7. Örnek Görüntü Karşılaştırması ──────────────────────────────────────
import cv2

pos_samples = manifest[manifest.label==1].sample(3, random_state=1)
neg_samples = manifest[manifest.label==0].sample(3, random_state=1)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Üst: Pozitif (PTX-498) | Alt: Negatif (NIH)', fontsize=13)

for ax, (_, row) in zip(axes[0], pos_samples.iterrows()):
    # merge görüntüsü varsa göster, yoksa img
    merge = str(row['image_path']).replace('.1.img.png', '.3.merge.png')
    show = merge if os.path.exists(merge) else row['image_path']
    ax.imshow(Image.open(show))
    ax.set_title(f"✅ PTX-498 | {row['case_id'][:15]}", fontsize=8)
    ax.axis('off')

for ax, (_, row) in zip(axes[1], neg_samples.iterrows()):
    ax.imshow(Image.open(row['image_path']), cmap='gray')
    ax.set_title(f"❌ NIH No Finding | {row['case_id'][:15]}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('✅ Dataset hazır — eğitime geçebilirsiniz!')